# Notebook para treinamento e testes de modelos

In [29]:
import pandas as pd
import numpy as np

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
import prophet
from model_generators import generate_ets_model, generate_sarimax_model, generate_prophet_model, feature_engineering, criar_ets_fipe_real, criar_sarimax_fipe_real, criar_prophet_fipe_real

from sklearn.model_selection import KFold
import matplotlib.pyplot as plt
import datetime
from dateutil.relativedelta import relativedelta
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
optuna.logging.set_verbosity(optuna.logging.ERROR)

In [40]:
horizonte_previsao = 3
tamanho_teste = 6
n_trials = 50
metrica_erro = 'mae'
tolerancia_fipe = 200
tolerancia_exog = 0.01

skus_teste = [0, 1, 2, 3, 4]

## Leitura da base de dados

In [31]:
# Base fipe historica
fipe_path = './data/dados_fipe_tratados.csv'
# Base da taxa de cambio
exchange_path = './data/DEXBZUS_tratados.csv'
# Base IPCA
ipca_path = './data/bcdata.sgs.433_tratados.csv' 

df_fipe = pd.read_csv(fipe_path)
print('FIPE shape:', df_fipe.shape)
print(df_fipe.head())

df_ex = pd.read_csv(exchange_path)
print('DEXBZUS head:')
print(df_ex.head())

df_ipca = pd.read_csv(ipca_path)
print('IPCA head:')
print(df_ipca.head())

FIPE shape: (411498, 9)
   Unnamed: 0       reference_date brand_name          model_name  year  \
0           0  2021-01-01 00:00:00       Fiat           147 C/ CL  1987   
1           1  2021-01-01 00:00:00       Fiat           147 C/ CL  1986   
2           2  2021-01-01 00:00:00       Fiat           147 C/ CL  1985   
3           3  2021-01-01 00:00:00       Fiat  147 Furgão (todos)  1987   
4           4  2021-01-01 00:00:00       Fiat  147 Furgão (todos)  1986   

  fuel_name  brl_price  year_of_reference month_of_reference  
0  Gasolina     2723.0               2021            January  
1  Gasolina     2484.0               2021            January  
2  Gasolina     2324.0               2021            January  
3  Gasolina     2199.0               2021            January  
4  Gasolina     2094.0               2021            January  
DEXBZUS head:
         date  exchange_rate
0  1995-01-01       0.846091
1  1995-02-01       0.841150
2  1995-03-01       0.890522
3  1995-04-01    

In [32]:
df_ipca['date'] = pd.to_datetime(df_ipca['date'])
df_ex['date'] = pd.to_datetime(df_ex['date'])

df_ipca.index = df_ipca['date']
df_ex.index = df_ex['date']

In [33]:
df_fipe = df_fipe.drop(columns=['Unnamed: 0'])

In [34]:
df_fipe['reference_date'] = pd.to_datetime(df_fipe['reference_date'], format='ISO8601')

df_fipe['sku'] = df_fipe.groupby(['brand_name', 'model_name', 'fuel_name', 'year']).ngroup()

In [35]:
# Retirar isso depois
df_fipe = df_fipe[df_fipe['reference_date'].dt.year > 2022]

In [36]:
df_fipe.head()

,reference_date,brand_name,model_name,year,fuel_name,brl_price,year_of_reference,month_of_reference,sku
125101,2023-09-01,Fiat,147 C/ CL,1987,Gasolina,4630.0,2023,September,2
125102,2023-09-01,Fiat,147 C/ CL,1986,Gasolina,4478.0,2023,September,1
125103,2023-09-01,Fiat,147 C/ CL,1985,Gasolina,3898.0,2023,September,0
125104,2023-09-01,Fiat,147 Furgão (todos),1987,Gasolina,2637.0,2023,September,5
125105,2023-09-01,Fiat,147 Furgão (todos),1986,Gasolina,2528.0,2023,September,4


In [37]:
df_fipe = df_fipe.drop(columns=['year_of_reference', 'month_of_reference'])

In [38]:
df_fipe.columns

Index(['reference_date', 'brand_name', 'model_name', 'year', 'fuel_name',
       'brl_price', 'sku'],
      dtype='str')

In [39]:
df_previsao_list = []

data_ref = pd.to_datetime('2026-04-01')

for sku in skus_teste:
    df = df_fipe.query("sku == @sku").copy()
    df = df.set_index('reference_date')

    meses_totais = relativedelta(data_ref, df.index.min()).years * 12 + relativedelta(data_ref, df.index.min()).months

    if meses_totais < 12:
        print(f"SKU: {sku} não tem dados suficientes ({meses_totais} meses apenas)")
        continue

    meses_esperados = pd.date_range(f'{df.index.min().year}-{df.index.min().month}', f'{data_ref.year}-{data_ref.month}', freq='MS')

    df = df.reindex(meses_esperados)

    df['brand_name'] = df['brand_name'].ffill()
    df['model_name'] = df['model_name'].ffill()
    df['year'] = df['year'].ffill()
    df['fuel_name'] = df['fuel_name'].ffill()

    df['brl_price'] = df['brl_price'].interpolate(method='linear')

    df = df.rename_axis('reference_date').reset_index()

    df.index = df['reference_date']
    df['sku'] = df['sku'].ffill()

    df_previsao_list.append(df)

df_previsao = pd.concat(df_previsao_list, ignore_index=True)

## Separar os dados em treino e teste para exógenas

In [ ]:
data_ref = pd.to_datetime('2026-04-01')
start = data_ref + relativedelta(months=-tamanho_teste)

train_ipca = df_ipca[df_ipca.index <= start]

test_ipca = df_ipca[df_ipca.index > start]

train_ex = df_ex[df_ex.index <= start]

test_ex = df_ex[df_ex.index > start]

exog_train = pd.concat([train_ipca['valor'], train_ex['exchange_rate']], axis=1)
exog_test = pd.concat([test_ipca['valor'], test_ex['exchange_rate']], axis=1)


df_prophet = pd.concat([df, df_ipca, df_ex], axis=1, sort=False)
df_prophet = df_prophet[['reference_date', 'date', 'brl_price', 'valor', 'exchange_rate']]
df_prophet = df_prophet.rename(columns={
    'reference_date': 'ds',
    'brl_price': 'y'
})

df_train_prophet = df_prophet[
    df_prophet['ds'] <= start
]
df_test_prophet = df_prophet[
    df_prophet['ds'] > start
]

df_train_prophet = df_train_prophet.reset_index(drop=True)
df_test_prophet = df_test_prophet.reset_index(drop=True)

## Geradores de modelos SARIMAX e ETS

## Modelos

#### Modelo do Câmbio

##### Prophet

In [ ]:
df_train_exchange = df_train_prophet[['ds', 'exchange_rate']].rename(columns={'exchange_rate': 'y'})
df_test_exchange = df_test_prophet[['ds', 'exchange_rate']].rename(columns={'exchange_rate': 'y'})

model_exchange, info_exchange_prophet, best_value_exchange_prophet = generate_prophet_model(
    df_train_exchange,
    df_test_exchange,
    [],
    n_trials,
    metrica_erro,
    tolerancia_exog
)

model_exchange.fit(
    df_train_exchange,
)

forecast_exchange = model_exchange.predict(
    df_test_exchange[['ds']]
)

pd.concat([df_test_exchange['y'], forecast_exchange['yhat']], axis=1)


##### SARIMAX

In [ ]:
model, info_exchange_sarimax, best_value_exchange_sarimax = generate_sarimax_model(train_ex['exchange_rate'], test_ex['exchange_rate'], None, None, n_trials, metrica_erro, tolerancia_exog)

results = model.fit(disp=False)

forecasts_ex_sarimax = results.forecast(steps=len(test_ex['exchange_rate']))

In [ ]:
print(f"Valor da métrica: {best_value_exchange_sarimax}")
display(pd.concat([forecasts_ex_sarimax, test_ex['exchange_rate']], axis=1))

#### Modelo do IPCA

##### Prophet

In [ ]:
df_train_ipca = df_train_prophet[['ds', 'valor']].rename(columns={'valor': 'y'})
df_test_ipca = df_test_prophet[['ds', 'valor']].rename(columns={'valor': 'y'})

df_train_prophet = df_train_prophet.ffill()
df_test_prophet = df_test_prophet.ffill()

model_ipca, info_ipca_prophet, best_value_ipca_prophet = generate_prophet_model(
    df_train_ipca,
    df_test_ipca,
    [],
    n_trials,
    metrica_erro,
    tolerancia_exog
)

model_ipca.fit(
    df_train_ipca
)

forecast_ipca = model_ipca.predict(
    df_test_ipca[['ds']]
)

pd.concat([df_test_ipca['y'], forecast_ipca['yhat']], axis=1)

##### SARIMAX

In [ ]:
model, info_ipca_sarimax, best_value_ipca_sarimax = generate_sarimax_model(train_ipca['valor'], test_ipca['valor'], None, None, n_trials, metrica_erro, tolerancia_exog)

results = model.fit(disp=False)

forecasts_ipca_sarimax = results.forecast(steps=len(test_ipca['valor']))

In [ ]:
print(f"Valor da métrica: {best_value_ipca_sarimax}")
display(pd.concat([forecasts_ipca_sarimax, test_ipca['valor']], axis=1))

#### Selecionar melhor forecast das exógenas

In [ ]:
metricas_ipca = np.array([best_value_ipca_prophet, best_value_ipca_sarimax])
metricas_exchange = np.array([best_value_exchange_prophet, best_value_exchange_sarimax])

idx_ipca = np.argmin(metricas_ipca)
idx_exchange = np.argmin(metricas_exchange)

forecast_ipca = pd.Series()
forecast_exchange = pd.Series()
modelo_escolhido_ipca = ''
modelo_escolhido_exchange = ''

if idx_ipca == 0:
    modelo_escolhido_ipca = 'Prophet'
    model_ipca = prophet.Prophet(**info_ipca_prophet)
    model_ipca.fit(pd.concat([df_train_ipca, df_test_ipca]))  
    future = model_ipca.make_future_dataframe(periods=horizonte_previsao, freq='MS')

    forecast_ipca = model_ipca.predict(
        future.tail(horizonte_previsao)
    )

    forecast_ipca.index = forecast_ipca['ds']
    forecast_ipca = forecast_ipca['yhat']
elif idx_ipca == 1:
    modelo_escolhido_ipca = 'SARIMAX'
    seasonal_order = (0, 0, 0, 0)

    if info_ipca_sarimax['seasonal']:
        seasonal_order = (
            info_ipca_sarimax['P'],
            info_ipca_sarimax['D'],
            info_ipca_sarimax['Q'],
            12
        )

    model_ipca = SARIMAX(
        pd.concat([train_ipca['valor'], test_ipca['valor']]),
        order=(
            info_ipca_sarimax['p'],
            info_ipca_sarimax['d'],
            info_ipca_sarimax['q']
        ),
        seasonal_order=seasonal_order,
        trend=info_ipca_sarimax['trend'],
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    results_ipca = model_ipca.fit(disp=False)

    forecast_ipca = results_ipca.forecast(steps=horizonte_previsao)

if idx_exchange == 0:
    modelo_escolhido_exchange = 'Prophet'
    model_exchange = prophet.Prophet(**info_exchange_prophet)
    model_exchange.fit(pd.concat([df_train_exchange, df_test_exchange]))  
    future = model_exchange.make_future_dataframe(periods=horizonte_previsao, freq='MS')

    forecast_exchange = model_exchange.predict(
        future.tail(horizonte_previsao)
    )

    forecast_exchange.index = forecast_exchange['ds']
    forecast_exchange = forecast_exchange['yhat']
elif idx_exchange == 1:
    modelo_escolhido_exchange = 'SARIMAX'
    seasonal_order = (0, 0, 0, 0)

    if info_exchange_sarimax['seasonal']:
        seasonal_order = (
            info_exchange_sarimax['P'],
            info_exchange_sarimax['D'],
            info_exchange_sarimax['Q'],
            12
        )

    model_exchange = SARIMAX(
        pd.concat([train_ex['exchange_rate'], test_ex['exchange_rate']]),
        order=(
            info_exchange_sarimax['p'],
            info_exchange_sarimax['d'],
            info_exchange_sarimax['q']
        ),
        seasonal_order=seasonal_order,
        trend=info_exchange_sarimax['trend'],
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    results_exchange = model_exchange.fit(disp=False)

    forecast_exchange = results_exchange.forecast(steps=horizonte_previsao)

forecast_exchange = forecast_exchange.rename("exchange_rate")
forecast_ipca = forecast_ipca.rename("valor")

exog_previsao = pd.concat([forecast_ipca, forecast_exchange], axis=1)

print(f"Modelo escolhido IPCA: {modelo_escolhido_ipca}")
print(f"Modelo escolhido taxa de câmbio: {modelo_escolhido_exchange}")

In [ ]:
display(forecast_exchange)

In [ ]:
display(forecast_ipca)

#### Modelo da FIPE

##### Separar os dados em treino e teste para fipe

In [ ]:
data_ref = pd.to_datetime('2026-04-01')
start = data_ref + relativedelta(months=-tamanho_teste)

train_ipca = df_ipca[df_ipca.index <= start]

test_ipca = df_ipca[df_ipca.index > start]

train_ex = df_ex[df_ex.index <= start]

test_ex = df_ex[df_ex.index > start]

exog_train = pd.concat([train_ipca['valor'], train_ex['exchange_rate']], axis=1)
exog_test = pd.concat([test_ipca['valor'], test_ex['exchange_rate']], axis=1)


df_prophet = pd.concat([df, df_ipca, df_ex], axis=1, sort=False)
df_prophet = df_prophet[['reference_date', 'date', 'brl_price', 'valor', 'exchange_rate']]
df_prophet = df_prophet.rename(columns={
    'reference_date': 'ds',
    'brl_price': 'y'
})

df_train_prophet = df_prophet[
    df_prophet['ds'] <= start
]
df_test_prophet = df_prophet[
    df_prophet['ds'] > start
]

df_train_prophet = df_train_prophet.reset_index(drop=True)
df_test_prophet = df_test_prophet.reset_index(drop=True)

In [ ]:
data_ref = pd.to_datetime('2026-04-01')
start = data_ref - relativedelta(months=tamanho_teste)

previsoes_por_sku = {}
modelo_vencedor_por_sku = {}

for sku in skus_teste:
    df_sku_atual = df_previsao.query('sku == @sku')

    train = df_sku_atual[
        df_sku_atual['reference_date'] <= start
    ]
    test = df_sku_atual[
        df_sku_atual['reference_date'] > start
    ]

    train.index = train['reference_date']
    test.index = test['reference_date']

    train = train[train['reference_date'] >= f'{data_ref.year - 5}-01-01']

    modelo_ets, best_value_ets, forecast_ets = criar_ets_fipe_real(train, test, n_trials, metrica_erro, tolerancia_fipe, horizonte_previsao)

    exog_train = exog_train[exog_train.index.isin(train['reference_date'])]

    modelo_sarimax, best_value_sarimax, forecast_sarimax = criar_sarimax_fipe_real(train, test, exog_train, exog_test, exog_previsao, n_trials, metrica_erro, tolerancia_fipe, horizonte_previsao)

    df_sku_atual.index = df_sku_atual['reference_date']
    df_prophet = pd.concat([df_sku_atual, pd.concat([exog_train, exog_test])], axis=1, sort=False)
    df_prophet = df_prophet[['reference_date', 'brl_price', 'valor', 'exchange_rate']]
    df_prophet = df_prophet.rename(columns={
        'reference_date': 'ds',
        'brl_price': 'y'
    })

    df_train_prophet = df_prophet[
        df_prophet['ds'] <= start
    ]
    df_test_prophet = df_prophet[
        df_prophet['ds'] > start
    ]

    df_train_prophet = df_train_prophet.reset_index(drop=True)
    df_test_prophet = df_test_prophet.reset_index(drop=True)
    modelo_prophet, best_value_prophet, forecast_prophet = criar_prophet_fipe_real(df_train_prophet, df_test_prophet, n_trials, metrica_erro, tolerancia_fipe, horizonte_previsao)

    resultados = {
        'ETS': {
            'erro': best_value_ets,
            'forecast': forecast_ets
        },
        'SARIMAX': {
            'erro': best_value_sarimax,
            'forecast': forecast_sarimax
        },
        'PROPHET': {
            'erro': best_value_prophet,
            'forecast': forecast_prophet
        }
    }

    melhor_modelo = min(
        resultados,
        key=lambda x: resultados[x]['erro']
    )

    previsoes_por_sku[sku] = resultados[melhor_modelo]['forecast']
    modelo_vencedor_por_sku[sku] = melhor_modelo

In [ ]:
previsoes_por_sku

In [ ]:
modelo_vencedor_por_sku

## Simulação da solução

A simulação consiste em estimar o lucro que seria obtido pelo cliente se usasse nossa solução para tomada de decisões de compra e venda para um conjunto de carros, para esse exemplo serão considerados 5 carros por um período de 6 meses, onde o modelo prevê um horizonte de um mês e o cliente toma a decisão para o esse mês com base na previsão, depois o modelo é retreinado e é simulado o próximo mês até bater os 6 meses.

### Política:

- Comprar o carro pelo preço atual se a previsão for de subida e for maior em pelo menos 0,1% do valor atual.
- Vender:
    - Pelo preço atual se a previsão for de queda e o valor for menor em pelo menos 0,1% do valor comprado.
    - E o preço atual for maior que o preço de compra.

### Valores iniciais
* Saldo: R$ 500.000
* Estoque de carros: 0 carros
* Horizonte de previsão: 1 mês

In [41]:
saldo = 500000
horizonte = 1

# Carro = (estoque, preco_compra)

skus = [100, 1832, 2134, 5112, 7023]
carros = {k: [] for k in skus}

start_sim = datetime.datetime(2025, 10, 1)
end_sim = start_sim + relativedelta(months=6)
curr_sim = start_sim

mensagens = ''

while curr_sim < end_sim:
    start = curr_sim + relativedelta(months=-3)
    # Treinar modelo para exogenas
    ## Separar dados de treino e teste
    train_ipca = df_ipca[df_ipca.index <= start]
    test_ipca = df_ipca[(df_ipca.index > start) & (df_ipca.index <= curr_sim)]

    train_ex = df_ex[df_ex.index <= start]
    test_ex = df_ex[(df_ex.index > start) & (df_ex.index <= curr_sim)]

    exog_train = pd.concat([train_ipca['valor'], train_ex['exchange_rate']], axis=1)
    exog_test = pd.concat([test_ipca['valor'], test_ex['exchange_rate']], axis=1)

    df_prophet = pd.concat([df, df_ipca, df_ex], axis=1, sort=False)
    df_prophet = df_prophet[['reference_date', 'date', 'brl_price', 'valor', 'exchange_rate']]
    df_prophet = df_prophet.rename(columns={
        'reference_date': 'ds',
        'brl_price': 'y'
    })

    df_train_prophet = df_prophet[
        df_prophet['ds'] <= start
    ]
    df_test_prophet = df_prophet[
        (df_prophet['ds'] > start) &
        (df_prophet['ds'] <= curr_sim)
    ]

    df_train_prophet = df_train_prophet.reset_index(drop=True)
    df_test_prophet = df_test_prophet.reset_index(drop=True)


    ## Treinar modelo do cambio
    ### Treinar modelo Sarimax
    model, info_exchange_sarimax, best_value_exchange_sarimax = generate_sarimax_model(train_ex['exchange_rate'], test_ex['exchange_rate'], None, None, n_trials, metrica_erro, tolerancia_exog)
    results = model.fit(disp=False)
    forecasts_ex_sarimax = results.forecast(steps=len(test_ex['exchange_rate']))

    ### Treinar modelo Prophet
    df_train_exchange = df_train_prophet[['ds', 'exchange_rate']].rename(columns={'exchange_rate': 'y'})
    df_test_exchange = df_test_prophet[['ds', 'exchange_rate']].rename(columns={'exchange_rate': 'y'})

    model_exchange, info_exchange_prophet, best_value_exchange_prophet = generate_prophet_model(
        df_train_exchange,
        df_test_exchange,
        [],
        n_trials,
        metrica_erro,
        tolerancia_exog
    )

    model_exchange.fit(
        df_train_exchange,
    )

    forecast_exchange = model_exchange.predict(
        df_test_exchange[['ds']]
    )

    ## Treinar modelo do IPCA
    ### Treinar modelo Sarimax
    model, info_ipca_sarimax, best_value_ipca_sarimax = generate_sarimax_model(train_ipca['valor'], test_ipca['valor'], None, None, n_trials, metrica_erro, tolerancia_exog)

    results = model.fit(disp=False)

    forecasts_ipca_sarimax = results.forecast(steps=len(test_ipca['valor']))

    ### Treinar modelo Prophet
    df_train_ipca = df_train_prophet[['ds', 'valor']].rename(columns={'valor': 'y'})
    df_test_ipca = df_test_prophet[['ds', 'valor']].rename(columns={'valor': 'y'})

    df_train_prophet = df_train_prophet.ffill()
    df_test_prophet = df_test_prophet.ffill()

    model_ipca, info_ipca_prophet, best_value_ipca_prophet = generate_prophet_model(
        df_train_ipca,
        df_test_ipca,
        [],
        n_trials,
        metrica_erro,
        tolerancia_exog
    )

    model_ipca.fit(
        df_train_ipca
    )

    forecast_ipca = model_ipca.predict(
        df_test_ipca[['ds']]
    )

    pd.concat([df_test_ipca['y'], forecast_ipca['yhat']], axis=1)

    # Selecionar o melhor modelo para cada exogena
    metricas_ipca = np.array([best_value_ipca_prophet, best_value_ipca_sarimax])
    metricas_exchange = np.array([best_value_exchange_prophet, best_value_exchange_sarimax])

    idx_ipca = np.argmin(metricas_ipca)
    idx_exchange = np.argmin(metricas_exchange)

    forecast_ipca = pd.Series()
    forecast_exchange = pd.Series()
    modelo_escolhido_ipca = ''
    modelo_escolhido_exchange = ''

    ultima_data = curr_sim

    future_dates = pd.date_range(
        ultima_data + relativedelta(months=1),
        periods=horizonte_previsao,
        freq='MS'
    )

    future = pd.DataFrame({'ds': future_dates})

    if idx_ipca == 0:
        modelo_escolhido_ipca = 'Prophet'
        model_ipca = prophet.Prophet(**info_ipca_prophet)
        model_ipca.fit(pd.concat([df_train_ipca, df_test_ipca]))

        forecast_ipca = model_ipca.predict(
            future
        )

        forecast_ipca.index = forecast_ipca['ds']
        forecast_ipca = forecast_ipca['yhat']
    elif idx_ipca == 1:
        modelo_escolhido_ipca = 'SARIMAX'
        seasonal_order = (0, 0, 0, 0)

        if info_ipca_sarimax['seasonal']:
            seasonal_order = (
                info_ipca_sarimax['P'],
                info_ipca_sarimax['D'],
                info_ipca_sarimax['Q'],
                12
            )

        model_ipca = SARIMAX(
            pd.concat([train_ipca['valor'], test_ipca['valor']]),
            order=(
                info_ipca_sarimax['p'],
                info_ipca_sarimax['d'],
                info_ipca_sarimax['q']
            ),
            seasonal_order=seasonal_order,
            trend=info_ipca_sarimax['trend'],
            enforce_stationarity=False,
            enforce_invertibility=False
        )

        results_ipca = model_ipca.fit(disp=False)

        forecast_ipca = results_ipca.forecast(steps=horizonte_previsao)

    if idx_exchange == 0:
        modelo_escolhido_exchange = 'Prophet'
        model_exchange = prophet.Prophet(**info_exchange_prophet)
        model_exchange.fit(pd.concat([df_train_exchange, df_test_exchange]))  

        forecast_exchange = model_exchange.predict(
            future
        )

        forecast_exchange.index = forecast_exchange['ds']
        forecast_exchange = forecast_exchange['yhat']
    elif idx_exchange == 1:
        modelo_escolhido_exchange = 'SARIMAX'
        seasonal_order = (0, 0, 0, 0)

        if info_exchange_sarimax['seasonal']:
            seasonal_order = (
                info_exchange_sarimax['P'],
                info_exchange_sarimax['D'],
                info_exchange_sarimax['Q'],
                12
            )

        model_exchange = SARIMAX(
            pd.concat([train_ex['exchange_rate'], test_ex['exchange_rate']]),
            order=(
                info_exchange_sarimax['p'],
                info_exchange_sarimax['d'],
                info_exchange_sarimax['q']
            ),
            seasonal_order=seasonal_order,
            trend=info_exchange_sarimax['trend'],
            enforce_stationarity=False,
            enforce_invertibility=False
        )

        results_exchange = model_exchange.fit(disp=False)

        forecast_exchange = results_exchange.forecast(steps=horizonte_previsao)

    forecast_exchange = forecast_exchange.rename("exchange_rate")
    forecast_ipca = forecast_ipca.rename("valor")

    exog_previsao = pd.concat([forecast_ipca, forecast_exchange], axis=1)

    print(f"Modelo escolhido IPCA: {modelo_escolhido_ipca}")
    print(f"Modelo escolhido taxa de câmbio: {modelo_escolhido_exchange}")

    # Selecionar apenas os dados contendo os skus da simulação com janela deslizante
    df_previsao_list = []

    for sku in skus:
        df = df_fipe.query("sku == @sku").copy()
        df = df.set_index('reference_date')

        meses_totais = relativedelta(curr_sim, df.index.min()).years * 12 + relativedelta(curr_sim, df.index.min()).months

        if meses_totais < 12:
            print(f"SKU: {sku} não tem dados suficientes ({meses_totais} meses apenas)")
            continue

        meses_esperados = pd.date_range(f'{df.index.min().year}-{df.index.min().month}', f'{curr_sim.year}-{curr_sim.month}', freq='MS')

        df = df.reindex(meses_esperados)

        df['brand_name'] = df['brand_name'].ffill()
        df['model_name'] = df['model_name'].ffill()
        df['year'] = df['year'].ffill()
        df['fuel_name'] = df['fuel_name'].ffill()

        df['brl_price'] = df['brl_price'].interpolate(method='linear')

        df = df.rename_axis('reference_date').reset_index()

        df.index = df['reference_date']
        df['sku'] = df['sku'].ffill()

        df_previsao_list.append(df)


    # Treinar 3 modelos para cada sku e escolher aquele com melhor desempenho
    df_skus = pd.concat(df_previsao_list, ignore_index=True)
    previsoes_por_sku = {}
    modelo_vencedor_por_sku = {}
    for sku in skus:
        df_sku_atual = df_skus.query('sku == @sku')

        train = df_sku_atual[
            df_sku_atual['reference_date'] <= start
        ]
        test = df_sku_atual[
            df_sku_atual['reference_date'] > start
        ]

        train.index = train['reference_date']
        test.index = test['reference_date']

        train = train[train['reference_date'] >= f'{curr_sim.year - 5}-01-01']

        modelo_ets, best_value_ets, forecast_ets = criar_ets_fipe_real(train, test, n_trials, metrica_erro, tolerancia_fipe, horizonte_previsao)

        exog_train = exog_train[exog_train.index.isin(train['reference_date'])]

        modelo_sarimax, best_value_sarimax, forecast_sarimax = criar_sarimax_fipe_real(train, test, exog_train, exog_test, exog_previsao, n_trials, metrica_erro, tolerancia_fipe, horizonte_previsao)

        df_sku_atual.index = df_sku_atual['reference_date']
        df_prophet = pd.concat([df_sku_atual, pd.concat([exog_train, exog_test])], axis=1, sort=False)
        df_prophet = df_prophet[['reference_date', 'brl_price', 'valor', 'exchange_rate']]
        df_prophet = df_prophet.rename(columns={
            'reference_date': 'ds',
            'brl_price': 'y'
        })

        df_train_prophet = df_prophet[
            df_prophet['ds'] <= start
        ]
        df_test_prophet = df_prophet[
            df_prophet['ds'] > start
        ]

        df_train_prophet = df_train_prophet.reset_index(drop=True)
        df_test_prophet = df_test_prophet.reset_index(drop=True)
        modelo_prophet, best_value_prophet, forecast_prophet = criar_prophet_fipe_real(df_train_prophet, df_test_prophet, n_trials, metrica_erro, tolerancia_fipe, horizonte_previsao)

        resultados = {
            'ETS': {
                'erro': best_value_ets,
                'forecast': forecast_ets
            },
            'SARIMAX': {
                'erro': best_value_sarimax,
                'forecast': forecast_sarimax
            },
            'PROPHET': {
                'erro': best_value_prophet,
                'forecast': forecast_prophet
            }
        }

        melhor_modelo = min(
            resultados,
            key=lambda x: resultados[x]['erro']
        )

        previsoes_por_sku[sku] = resultados[melhor_modelo]['forecast']
        modelo_vencedor_por_sku[sku] = melhor_modelo

    # Política do cliente
    for sku, carro in carros.items():
        preco_atual = df_skus.query('sku == @sku and reference_date == @curr_sim')['brl_price'].iloc[0]
        preco_previsto = previsoes_por_sku[sku].iloc[0]

        # Comprar
        if preco_previsto >= preco_atual*1.001 and saldo >= preco_atual:
            mensagem = f'Comprou o sku {sku} por {preco_atual}\n'
            print(mensagem)
            mensagens += mensagem
            saldo -= preco_atual
            carros[sku].append((1, preco_atual))
        
        # Vender
        for idx, (estoque, compra) in enumerate(carro):
            if (preco_previsto <= compra*0.999) and (preco_atual > compra):
                mensagem = f'Vendeu o sku {sku} por {preco_atual}\n'
                mensagens += mensagem
                print(mensagem)
                saldo += preco_atual
                carros[sku].pop(idx)

        
    curr_sim += relativedelta(months=1)
    horizonte_previsao += 1

  0%|          | 0/50 [00:00<?, ?it/s]c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
Best trial: 0. Best value: 0.168725:   2%|▏         | 1/50 [00:00<00:07,  6.56it/s]c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency 

Modelo escolhido IPCA: SARIMAX
Modelo escolhido taxa de câmbio: Prophet


  0%|          | 0/50 [00:00<?, ?it/s]c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
Best trial: 0. Best value: 697.186:   0%|          | 0/50 [00:00<?, ?it/s]c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
Best trial: 1. Best value: 185.75:   4%|▍         | 2/50 [00:00<00:03, 13.62it/s]
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: Va

                  simulation  brl_price
reference_date                         
2025-08-01      69014.783443    69062.0
2025-09-01      69423.659400    69154.0
2025-10-01      68720.467003    69154.0
185.75024491899725
2025-11-01    69256.465256
2025-12-01    69292.867427
2026-01-01    69610.477243
Freq: MS, Name: simulation, dtype: float64


  0%|          | 0/50 [00:00<?, ?it/s]c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
Best trial: 0. Best value: 722.306:   2%|▏         | 1/50 [00:00<00:06,  7.29it/s]c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
Best trial: 1. Best value: 343.375:   4%|▍         | 2/50 

...

                  simulation  brl_price
reference_date                         
2026-01-01      43263.638699    43549.0
2026-02-01      43338.043502    43027.0
2026-03-01      42934.476710    42766.0
274.44126972388887
2026-04-01    43298.805021
2026-05-01    43704.335222
2026-06-01    43652.007880
2026-07-01    43573.322244
2026-08-01    43259.163744
2026-09-01    43854.410666
2026-10-01    43373.099593
2026-11-01    43152.450009
Freq: MS, Name: simulation, dtype: float64


Best trial: 0. Best value: 539.732:   0%|          | 0/50 [00:00<?, ?it/s]c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
Best trial: 1. Best value: 312.875:   8%|▊         | 4/50 [00:00<00:03, 14.64it/s]c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'
Best trial: 1. Best value: 312.875:   8%|▊         | 4/50 [00:00<00:03, 14.64it/s]c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
 

                predicted_mean  brl_price
reference_date                           
2026-01-01        43147.076237    43549.0
2026-02-01        42975.533227    43027.0
2026-03-01        43140.451379    42766.0
280.52603564025776
2026-04-01    42678.844254
2026-05-01    43051.635830
2026-06-01    42575.823365
2026-07-01    42536.797131
2026-08-01    42505.104804
2026-09-01    42759.176435
2026-10-01    42464.338640
2026-11-01    42450.973622
Freq: MS, Name: predicted_mean, dtype: float64


  0%|          | 0/50 [00:00<?, ?it/s]21:20:49 - cmdstanpy - INFO - Chain [1] start processing
21:20:49 - cmdstanpy - INFO - Chain [1] done processing
Best trial: 0. Best value: 1283.26:   2%|▏         | 1/50 [00:00<00:13,  3.74it/s]21:20:50 - cmdstanpy - INFO - Chain [1] start processing
21:20:50 - cmdstanpy - INFO - Chain [1] done processing
Best trial: 1. Best value: 399.497:   4%|▍         | 2/50 [00:00<00:13,  3.50it/s]21:20:50 - cmdstanpy - INFO - Chain [1] start processing
21:20:50 - cmdstanpy - INFO - Chain [1] done processing
Best trial: 1. Best value: 399.497:   6%|▌         | 3/50 [00:00<00:12,  3.72it/s]21:20:50 - cmdstanpy - INFO - Chain [1] start processing
21:20:50 - cmdstanpy - INFO - Chain [1] done processing
Best trial: 1. Best value: 399.497:   8%|▊         | 4/50 [00:01<00:13,  3.53it/s]21:20:50 - cmdstanpy - INFO - Chain [1] start processing
21:20:51 - cmdstanpy - INFO - Chain [1] done processing
Best trial: 1. Best value: 399.497:  10%|█         | 5/50 [00:01<00:1

260.94376244799304
         y          yhat
0  43549.0  42684.624018
1  43027.0  43003.008608
2  42766.0  42983.767937
ds
2026-04-01    43370.061341
2026-05-01    43420.941973
2026-06-01    43570.237174
2026-07-01    44563.898514
2026-08-01    44425.117076
2026-09-01    44518.203825
2026-10-01    43866.083487
2026-11-01    44379.986252
Name: yhat, dtype: float64


In [42]:
saldo

np.float64(11987.0)

In [43]:
carros

{100: [(1, np.float64(69154.0))],
 1832: [],
 2134: [(1, np.float64(98402.0)),
  (1, np.float64(97061.0)),
  (1, np.float64(95978.0))],
 5112: [],
 7023: [(1, np.float64(42601.0)),
  (1, np.float64(42477.0)),
  (1, np.float64(42696.0))]}

In [45]:
11987.0+69154.0+98402.0+97061.0+95978.0+42601.0+42477.0+42696.0

500356.0

In [49]:
66503.0-66356.0 + 66611.0-66402.0

356.0

In [44]:
print(mensagens)

Comprou o sku 100 por 69154.0
Comprou o sku 7023 por 42601.0
Comprou o sku 1832 por 66356.0
Comprou o sku 2134 por 98402.0
Comprou o sku 7023 por 42477.0
Vendeu o sku 1832 por 66503.0
Comprou o sku 7023 por 42696.0
Comprou o sku 1832 por 66402.0
Comprou o sku 2134 por 97061.0
Vendeu o sku 1832 por 66611.0
Comprou o sku 2134 por 95978.0

